# Wake-word evaluation: full pipeline

Runs the trained wake/sleep classifiers over the recorded clips in `~/jarvie-data/audio/` and
measures how well they work:

1. **Featurize**: 16 kHz audio → livekit-wakeword's mel spectrogram → 96-dim speech embeddings (one per 80 ms)
2. **Score**: slide the classifier's 16-frame window (~1.96 s: each embedding covers 0.76 s, one every 80 ms) over each clip → one score per 80 ms
3. **Metrics**: recall / false-rejection rate (FRR) on positives, false accepts per hour (FA/h) on negatives,
   swept across thresholds
4. **Operating point**: the highest-recall threshold within a false-accept budget
5. **Error analysis**: which positions/conditions and which individual clips fail
6. **Negatives by similarity**: section 11 breaks false accepts down by how close each negative
   sounds to the wake phrase (not similar, similar, very similar)
7. **Pace comparison**: normal pace is the **primary** result (sections 3-9); section 10 compares it with
   fast, slow, extreme (fast + slow) and composite (all paces)

Clip names follow `<kind>_<position>_<condition>_<phrase>_<index>.wav`, e.g. `pos_desk_quiet_wake_003.wav`
(recorded with `scratch/record.py`). Speaking pace is folded into the condition: `quiet-normal`,
`quiet-fast` or `quiet-slow` (a plain `quiet` with no pace also counts as normal).
Negatives also carry a similarity tier in the kind field: `neg-not-similar`, `neg-similar` or
`neg-very-similar` (a plain `neg` counts as `unknown`).

Each clip is padded with 1 s of silence on both sides before scoring (`PAD_SECONDS`), as livekit's own
`predict_clip` does; times in the plots are relative to the start of the recording.

**Kernel:** use the project environment, `server/.venv` (*pipecat-quickstart*). The feature models ship with the
`livekit-wakeword` package, preprocessed exactly as in training.

In [ ]:
# Re-import edited project modules (e.g. jarvie/eval/wakeword.py) on every run, so the
# notebook never keeps using an old copy from before a code change.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Find server/ (the folder with models/ and jarvie/) from wherever the notebook was started,
# and put it on sys.path so `import jarvie` works.
SERVER = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models").is_dir() and (p / "jarvie").is_dir())
sys.path.insert(0, str(SERVER))

from jarvie.eval import wakeword

pd.set_option("display.precision", 3)
print("server dir:", SERVER)
print("wakeword module:", wakeword.__file__)

## Configuration
`EXCLUDE` lists clips known to be mislabeled (from the notes at the bottom of `scratch/record.py`).
Clips with loud background are kept: they are valid negatives, and exactly what the model has to reject.

In [ ]:
import os

# audio/<phrase>/<clip>.wav; JARVIE_AUDIO_DIR overrides it (e.g. to evaluate a held-out set)
AUDIO_DIR = Path(os.environ.get("JARVIE_AUDIO_DIR", Path.home() / "jarvie-data" / "audio"))
MODELS = {
    "wake": SERVER / "models" / "jarvie_wake.onnx",
    "sleep": SERVER / "models" / "jarvie_sleep.onnx",
}
HOP = 1                                  # score every embedding frame (80 ms)
PAD_SECONDS = 1.0                        # silence added before/after each clip (as livekit's predict_clip)
THRESHOLDS = np.linspace(0, 1, 101)
MAX_FPPH = 0.1                           # false-accept budget: at most one every 10 hours

EXCLUDE: set[str] = {
    # "neg_desk_quiet_wake_014.wav",   # e.g. a negative that actually says the wake word
}

# Pace sets compared in section 10. Each set is evaluated within its own group: positives AND
# negatives spoken at those paces (fast positives vs fast negatives, ...). Set this to False to
# pair every set with all negatives instead, so only recall differs between sets.
MATCH_NEGATIVE_PACE = True
PRIMARY = "primary (normal)"
PACE_SETS = {
    PRIMARY: ["normal"],
    "fast": ["fast"],
    "slow": ["slow"],
    "extreme (fast + slow)": ["fast", "slow"],
    "composite (all paces)": ["fast", "normal", "slow"],
}

## 1. Dataset overview

In [ ]:
clips = pd.DataFrame(
    [{"phrase": d.name, "file": p.name} for d in sorted(AUDIO_DIR.iterdir()) if d.is_dir()
     for p in sorted(d.glob("*.wav"))]
)
if clips.empty:
    raise SystemExit(f"no clips found under {AUDIO_DIR}")

parts = clips.file.str.removesuffix(".wav").str.split("_", expand=True)
kind = parts[0].str.split("-", n=1, expand=True).reindex(columns=[0, 1])
clips["kind"], clips["position"] = kind[0], parts[1]
clips["similarity"] = kind[1].where(clips.kind == "neg").fillna("")
clips.loc[(clips.kind == "neg") & (clips.similarity == ""), "similarity"] = "unknown"
cond = parts[2].str.split("-", n=1, expand=True).reindex(columns=[0, 1])
clips["condition"], clips["pace"] = cond[0], cond[1].fillna("normal")
clips["group"] = np.where(clips.kind == "pos", "pos", "neg: " + clips.similarity)
clips.pivot_table(index=["phrase", "position", "condition", "pace"], columns="group",
                  values="file", aggfunc="count", fill_value=0)

## 2. Featurize and score
`make_featurizer()` builds the audio → embedding step; `load_model()` returns the classifier, its input name
and its window length (16 frames). Each phrase is scored with its own model. Sections 3-9 and 11 use `primary`: normal-pace positives and normal-pace negatives
(all negatives if `MATCH_NEGATIVE_PACE` is off).

In [ ]:
featurize = wakeword.make_featurizer()

frames = []
for phrase, model_path in MODELS.items():
    folder = AUDIO_DIR / phrase
    if not model_path.exists() or not any(folder.glob("*.wav")):
        print(f"{phrase}: skipped (model or clips missing)")
        continue
    session, input_name, window = wakeword.load_model(model_path)
    scored = wakeword.score_all(session, folder, input_name, window, HOP, featurize, PAD_SECONDS)
    print(f"{phrase}: scored {len(scored)} clips (window = {window} frames = {0.76 + (window - 1) * 0.08:.2f} s)")
    frames.append(scored)

results = pd.concat(frames, ignore_index=True)
excluded = results[results.file.isin(EXCLUDE)]
results = results[~results.file.isin(EXCLUDE)].reset_index(drop=True)
print(f"excluded {len(excluded)} clip(s):", list(excluded.file))
PHRASES = list(results.phrase.unique())

# record.py folds pace into the condition ("quiet-fast"); split it back out. No suffix = normal.
cond = results.condition.str.split("-", n=1, expand=True).reindex(columns=[0, 1])
results["condition"], results["pace"] = cond[0], cond[1].fillna("normal")

# Negatives carry a similarity tier in the kind field ("neg-similar"); split it out too.
kind = results.kind.str.split("-", n=1, expand=True).reindex(columns=[0, 1])
results["kind"] = kind[0]
results["similarity"] = kind[1].where(results.kind == "neg").fillna("")
results.loc[(results.kind == "neg") & (results.similarity == ""), "similarity"] = "unknown"


def pace_subset(d: pd.DataFrame, paces: list[str]) -> pd.DataFrame:
    """Clips spoken at the given paces: positives, and negatives too if MATCH_NEGATIVE_PACE
    (otherwise all negatives, whatever their pace)."""
    if MATCH_NEGATIVE_PACE:
        return d[d.pace.isin(paces)]
    return d[(d.kind == "neg") | d.pace.isin(paces)]


primary = pace_subset(results, PACE_SETS[PRIMARY])
if primary[primary.kind == "pos"].empty:
    raise SystemExit("no normal-pace positives: record some, or point PRIMARY at another pace set")
results.groupby(["phrase", "kind", "similarity", "pace"]).size().rename("clips").to_frame()

## 3. Score distributions (primary: normal pace)
Each clip's **max score** is what decides a detection. A usable model puts positives (blue) to the right of
negatives (orange) with little overlap; the overlap is where errors come from at any threshold.

In [ ]:
fig, axes = plt.subplots(1, len(PHRASES), figsize=(6 * len(PHRASES), 3.5), squeeze=False)
bins = np.linspace(0, 1, 21)
for ax, phrase in zip(axes[0], PHRASES):
    d = primary[primary.phrase == phrase]
    ax.hist(d[d.kind == "pos"].max_score, bins=bins, alpha=0.6, label="positive")
    ax.hist(d[d.kind == "neg"].max_score, bins=bins, alpha=0.6, label="negative")
    ax.set(title=f"{phrase}: max score per clip", xlabel="max score", ylabel="clips")
    ax.legend()
plt.tight_layout()
plt.show()

primary.groupby(["phrase", "kind"]).max_score.describe()[["count", "mean", "min", "50%", "max"]]

## 4. Threshold sweep (primary: normal pace)
`sweeps()` computes, at each threshold, **recall** on positives and **false accepts per hour** on negatives
(consecutive above-threshold frames count as one event). The **DET curve** (FRR vs FA/h) is the standard
wake-word trade-off plot: lower-left is better.

Also shown is the clip-level **equal error rate (EER)**: the threshold where the fraction of positives missed
equals the fraction of negative clips that trigger. It's a single-number summary that doesn't depend on how
many hours of negatives were recorded.

In [ ]:
def eer_point(d: pd.DataFrame, curve: pd.DataFrame) -> tuple[float, float]:
    """(threshold, rate) where the share of positives missed ~= the share of negative clips that fire."""
    neg_trigger = np.array([(d[d.kind == "neg"].max_score >= t).mean() for t in THRESHOLDS])
    gap = np.abs(curve.frr.to_numpy() - neg_trigger)
    if np.isnan(gap).all():
        return float("nan"), float("nan")
    i = int(np.nanargmin(gap))
    return float(THRESHOLDS[i]), float((curve.frr.iloc[i] + neg_trigger[i]) / 2)


curves, eer = {}, {}
for phrase in PHRASES:
    d = primary[primary.phrase == phrase]
    curves[phrase] = curve = wakeword.sweeps(d, THRESHOLDS, phrase=phrase)
    eer[phrase] = eer_point(d, curve)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for phrase, curve in curves.items():
    axes[0].plot(curve.threshold, curve.recall, label=f"{phrase} recall")
    axes[0].plot(curve.threshold, curve.fpph / curve.fpph.max(), "--", label=f"{phrase} FA/h (scaled)")
    axes[1].plot(curve.fpph, curve.frr, marker=".", label=phrase)
axes[0].set(title="Recall and false accepts vs threshold", xlabel="threshold")
axes[1].set(title="DET curve", xlabel="false accepts per hour", ylabel="false rejection rate", xscale="symlog")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

for phrase, (t, rate) in eer.items():
    print(f"{phrase}: EER ≈ {rate:.1%} at threshold {t:.2f}")

## 5. Operating point (chosen on the primary set)
The threshold is chosen once, on normal pace, and reused unchanged in section 10, the way
the deployed detector would use one threshold for every way you say the phrase.

**Measurement resolution:** with *H* hours of negative audio, one false accept already means 1/*H* FA/h,
so "0 false accepts" only proves FA/h < 1/*H*. Checking a budget of `MAX_FPPH` therefore needs at least
1/`MAX_FPPH` hours of negatives (10 h for 0.1 FA/h) of everyday background audio (talking, music, video).

- **Enough negatives:** `operating_point()` picks the highest-recall threshold with FA/h ≤ `MAX_FPPH`.
- **Not enough (the usual case with short clips):** the budget can't be checked, and enforcing it would just
  pick the lowest threshold above every negative's score, with near-zero recall. The **EER threshold** is used
  instead; the budget-based threshold is still printed for reference.

In [ ]:
chosen = {}
for phrase, curve in curves.items():
    neg_hours = curve.neg_hours.iloc[0]
    needed_hours = 1 / MAX_FPPH
    print(f"{phrase}: {neg_hours * 60:.1f} min of negatives -> one false accept = {1 / neg_hours:.0f} FA/h "
          f"(need {needed_hours:.0f} h to check {MAX_FPPH} FA/h)")

    try:
        point = wakeword.operating_point(curve, max_fpph=MAX_FPPH)
        budget_note = (f"budget threshold {point.threshold:.2f} -> recall {point.recall:.1%}, "
                       f"{point.false_events:.0f} false accepts")
    except ValueError as e:
        point, budget_note = None, str(e)

    if neg_hours >= needed_hours and point is not None and point.recall > 0:
        chosen[phrase] = float(point.threshold)
        print(f"  operating point: {budget_note}, {point.fpph:.2f} FA/h")
    else:
        # Too little negative audio to verify the budget: "0 false accepts" in a few minutes says
        # nothing about 10 hours, so use the EER threshold, which doesn't depend on duration.
        chosen[phrase] = float(eer[phrase][0])
        print(f"  not enough negatives to check the budget ({budget_note}); "
              f"using the EER threshold {chosen[phrase]:.2f}")

## 6. Recall by position and condition (primary: normal pace)
At the chosen threshold, where does the model miss? If recall is high at the desk but low when loud,
the fix is more noisy training data, not a different model.

In [ ]:
for phrase, threshold in chosen.items():
    print(f"{phrase} @ threshold {threshold:.2f}")
    display(wakeword.by_position_and_condition(primary, threshold, phrase))

## 7. Error analysis (primary: normal pace)
The positives the model scores lowest (likely misses) and the negatives it scores highest (likely false
accepts). Listen to these first: they're either labeling mistakes or the cases to add training data for.

In [ ]:
cols = ["file", "position", "condition", "pace", "similarity", "max_score"]
for phrase in PHRASES:
    d = primary[primary.phrase == phrase]
    print(f"{phrase}: hardest positives")
    display(d[d.kind == "pos"].nsmallest(5, "max_score")[cols])
    print(f"{phrase}: highest-scoring negatives")
    display(d[d.kind == "neg"].nlargest(5, "max_score")[cols])

## 8. Score over time (primary: normal pace)
Frame-by-frame scores for the extreme clips. A clean detection is a single peak above the threshold while
the phrase is spoken; a negative that crosses the line shows when in the clip it fired.

In [ ]:
for phrase in PHRASES:
    d = primary[primary.phrase == phrase]
    picks = pd.concat([d[d.kind == "pos"].nlargest(2, "max_score"), d[d.kind == "pos"].nsmallest(1, "max_score"),
                       d[d.kind == "neg"].nlargest(2, "max_score")])
    fig, ax = plt.subplots(figsize=(10, 3.5))
    for _, row in picks.iterrows():
        t = np.arange(len(row.frame_scores)) * 0.08 * HOP - PAD_SECONDS  # 0 = start of the recording
        ax.plot(t, row.frame_scores, ls="-" if row.kind == "pos" else "--", label=row.file)
    ax.axhline(chosen[phrase], color="gray", lw=1, label=f"threshold {chosen[phrase]:.2f}")
    ax.set(title=f"{phrase}: score over time", xlabel=f"window start (s); each window spans {0.76 + (window - 1) * 0.08:.2f} s",
           ylabel="score", ylim=(0, 1))
    ax.legend(fontsize=8, loc="upper right")
    plt.tight_layout()
    plt.show()

## 9. Summary (primary: normal pace)

In [ ]:
summary = []
for phrase, threshold in chosen.items():
    row = curves[phrase].iloc[int(np.argmin(np.abs(THRESHOLDS - threshold)))]
    summary.append({"phrase": phrase, "threshold": threshold, "recall": row.recall, "frr": row.frr,
                    "false_events": row.false_events, "fa_per_hour": row.fpph,
                    "neg_minutes": row.neg_hours * 60, "eer": eer[phrase][1]})
pd.DataFrame(summary).set_index("phrase")

In [ ]:
# When in each clip does the score peak? Peak frame i is the window covering
# i*0.08 s to i*0.08 + ~1.96 s (each embedding covers 0.76 s; 16 of them, 80 ms apart). If both kinds mostly peak at frame 0, the model is reacting
# to the start of the recording (e.g. the Enter-key click), not the spoken phrase.
FRAME_S = 0.08 * HOP
WINDOW_S = 0.76 + (window - 1) * 0.08


def peak_table(d: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    groups = {
        "worst negatives (highest max)": d[d.kind == "neg"].nlargest(n, "max_score"),
        "worst positives (lowest max)": d[d.kind == "pos"].nsmallest(n, "max_score"),
    }
    rows = []
    for group, picks in groups.items():
        for _, r in picks.iterrows():
            if not r.frame_scores:  # clip shorter than one window
                continue
            peak = int(np.argmax(r.frame_scores))
            start = peak * FRAME_S - PAD_SECONDS  # relative to the recording's start
            rows.append({
                "group": group,
                "file": r.file,
                "max_score": r.max_score,
                "peak_frame": peak,
                "peak_window": f"{start:.2f}-{start + WINDOW_S:.2f} s",
                "n_frames": len(r.frame_scores),
            })
    return pd.DataFrame(rows).set_index(["group", "file"])


for phrase in PHRASES:
    d = primary[primary.phrase == phrase]
    print(f"{phrase}: peaks of the worst clips")
    display(peak_table(d))

    # Across all clips: how often is the peak in the very first window?
    first = d.frame_scores.apply(lambda s: bool(s) and int(np.argmax(s)) == 0)
    print(f"{phrase}: clips peaking in the first window ->",
          {k: f"{v:.0%}" for k, v in first.groupby(d.kind).mean().items()})

## 10. Pace comparison
Each pace set is evaluated at the **primary threshold** (chosen on normal pace in section 5), so the
numbers show what the deployed detector would do if you spoke that way:

- **primary (normal)**: the reference
- **fast**, **slow**: each pace on its own
- **extreme (fast + slow)**: how much worse off-normal speech is overall
- **composite (all paces)**: the realistic mix

Each set is compared **within its own group**: its positives against negatives spoken at the same
pace(s), so `n_neg`, false accepts and EER differ per row. Keep in mind that groups can differ in how hard
their negatives are (check the similarity mix in section 1): a pace whose negatives are all *not similar*
will look better on false accepts than one with *very similar* negatives. With `MATCH_NEGATIVE_PACE = False`
every set uses all negatives instead, and only recall differs. `Δ recall` is relative to the primary set: a large negative value means that pace needs
more training examples. EER is computed per set at its own best threshold, for a threshold-free view.

In [ ]:
rows = []
for phrase in PHRASES:
    d = results[results.phrase == phrase]
    threshold = chosen[phrase]
    t_i = int(np.argmin(np.abs(THRESHOLDS - threshold)))
    for name, paces in PACE_SETS.items():
        s = pace_subset(d, paces)
        n_pos = int((s.kind == "pos").sum())
        row = {"phrase": phrase, "set": name, "n_pos": n_pos, "n_neg": int((s.kind == "neg").sum())}
        if n_pos:
            curve = wakeword.sweeps(s, THRESHOLDS, phrase=phrase)
            at_t = curve.iloc[t_i]
            row.update(threshold=threshold, recall=at_t.recall, frr=at_t.frr,
                       fa_per_hour=at_t.fpph, eer=eer_point(s, curve)[1])
        rows.append(row)

pace_table = pd.DataFrame(rows).set_index(["phrase", "set"])
primary_recall = pace_table.xs(PRIMARY, level="set").recall
pace_table["Δ recall"] = pace_table.recall - primary_recall.reindex(pace_table.index, level="phrase")
pace_table

In [ ]:
# Recall vs threshold for each individual pace, with the primary threshold marked; and where each
# pace misses, broken down by position and condition.
for phrase in PHRASES:
    d = results[results.phrase == phrase]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for pace in ["normal", "fast", "slow"]:
        s = pace_subset(d, [pace])
        if (s.kind == "pos").any():
            curve = wakeword.sweeps(s, THRESHOLDS, phrase=phrase)
            ax.plot(curve.threshold, curve.recall, label=f"{pace} (n={int((s.kind == 'pos').sum())})")
    ax.axvline(chosen[phrase], color="gray", lw=1, ls="--", label=f"primary threshold {chosen[phrase]:.2f}")
    ax.set(title=f"{phrase}: recall by speaking pace", xlabel="threshold", ylabel="recall", ylim=(0, 1.02))
    ax.legend()
    plt.tight_layout()
    plt.show()

    pos = d[d.kind == "pos"].assign(found=lambda x: x.max_score >= chosen[phrase])
    print(f"{phrase}: recall at the primary threshold, by position/condition (rows) and pace (columns)")
    display(pos.pivot_table(index=["position", "condition"], columns="pace", values="found",
                            aggfunc="mean").reindex(columns=["normal", "fast", "slow"]))

## 11. Negatives by similarity tier
How the model handles negatives of increasing difficulty, at the primary threshold:

- **not similar**: everyday speech; any false accept here is a serious problem
- **similar**: words that partly sound like the wake phrase
- **very similar**: near-misses such as "jacky" for "jarvie"; the hardest case

`clips_firing` is the share of that tier's clips whose max score crosses the threshold. `eer_vs_positives`
pits normal-pace positives against only that tier, a threshold-free measure of how separable the two are:
it should rise from not similar to very similar, and a high value for *not similar* means the model is
reacting to something other than the phrase (e.g. the clip onset).

In [ ]:
TIERS = ["not-similar", "similar", "very-similar", "unknown"]
rows = []
for phrase in PHRASES:
    d = primary[primary.phrase == phrase]   # normal-pace positives + normal-pace negatives
    pos = d[d.kind == "pos"]
    threshold = chosen[phrase]
    for tier in TIERS:
        neg = d[(d.kind == "neg") & (d.similarity == tier)]
        if neg.empty:
            continue
        both = pd.concat([pos, neg])
        curve = wakeword.sweeps(both, THRESHOLDS, phrase=phrase)
        rows.append({
            "phrase": phrase,
            "tier": tier,
            "n_neg": len(neg),
            "mean_max_score": neg.max_score.mean(),
            "worst_max_score": neg.max_score.max(),
            "clips_firing": (neg.max_score >= threshold).mean(),
            "false_events": sum(wakeword.event_count(s, threshold) for s in neg.frame_scores),
            "eer_vs_positives": eer_point(both, curve)[1],
        })
tier_table = pd.DataFrame(rows).set_index(["phrase", "tier"])
tier_table

In [ ]:
# Max score per clip: normal-pace positives next to each negative tier, with the primary threshold.
for phrase in PHRASES:
    d = primary[primary.phrase == phrase]
    groups = {"positives (normal)": d[d.kind == "pos"].max_score}
    for tier in TIERS:
        neg = d[(d.kind == "neg") & (d.similarity == tier)]
        if not neg.empty:
            groups[f"neg: {tier}"] = neg.max_score

    fig, ax = plt.subplots(figsize=(8, 3.5))
    rng = np.random.default_rng(0)
    for x, (name, scores) in enumerate(groups.items()):
        jitter = rng.uniform(-0.15, 0.15, len(scores))   # spread points so overlaps stay visible
        ax.scatter(x + jitter, scores, s=18, alpha=0.7)
        ax.hlines(scores.median(), x - 0.25, x + 0.25, color="black", lw=1)
    ax.axhline(chosen[phrase], color="gray", ls="--", lw=1, label=f"primary threshold {chosen[phrase]:.2f}")
    ax.set_xticks(range(len(groups)), list(groups), rotation=15)
    ax.set(title=f"{phrase}: max score by group (black bar = median)", ylabel="max score", ylim=(0, 1))
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()